# 1.Basic Data Exploration
Load and understand your data.

understanding your data is the first part of model training.
**Using Pandas to Get Familiar With Your Data**
pandas is the major library to work with data.It works on DataFrames that are just like tables in Excel.

In [131]:
import pandas as pd

As an example we will start with [text](data/melb_data.csv) this housing dataset.

In [132]:
file_path = "../data/melb_data.csv"
property_data = pd.read_csv(file_path)
property_data.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


Reading the data , first row of count mean how much non_zero counts each column have.
we can get missing values due to several reasons, we have to deal with them before moving to model training.

# 2.Your First Machine Learning Model
we have too many variables in data , we dont need alll to work with.
**Selecting Data for Modeling**
to see the variables we use columns property of pandas to get list of our columns.

In [133]:
property_data.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='object')

we need just these columns ['Rooms', 'Bedroom2' 'Bathroom', 'Suburb', 'Distance', 'Postcode', 'BuildingAres', 'Landsize']
and these have no empty value.

we have 7130 non-zero values in BuildingArea , 8205 in yearbuilt and 13518 in car.we will handel them and fill them.

In [134]:
# 1. Drop unused columns
property_data = property_data.drop(columns=['Address','Type','Method','SellerG','Date','CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'])

# 2. Handle 'Car' missing values using the mode
car_mode = property_data["Car"].mode()[0]
property_data["Car"] = property_data["Car"].fillna(car_mode)

# 3. Handle 'BuildingArea' and 'YearBuilt' using their respective medians
area_median = property_data["BuildingArea"].median()
property_data["BuildingArea"] = property_data["BuildingArea"].fillna(area_median)

year_median = property_data["YearBuilt"].median()
property_data["YearBuilt"] = property_data["YearBuilt"].fillna(year_median) # FIXED: Changed area_median to year_median

# 4. Drop rows with missing Price and save it back to the variable
property_data = property_data.dropna(subset=['Price'])

# 5. Check your final cleaned data stats!
property_data.describe()


,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.611856,558.416127,139.633972,1966.788218
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.960793,3990.669241,392.217403,29.088642
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,122.000000,1960.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,129.940000,1975.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000


**Selecting The Prediction Target**

In [135]:
y = property_data.Price #our model prediction target

**Choosing "Features"**

In [136]:
features = ['Rooms', 'Bedroom2', 'Bathroom', 'Distance', 'Postcode', 'BuildingArea', 'Landsize', 'Car', 'YearBuilt']
X = property_data[features] # model features
X.describe()

,Rooms,Bedroom2,Bathroom,Distance,Postcode,BuildingArea,Landsize,Car,YearBuilt
count,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,2.914728,1.534242,10.137776,3105.301915,139.633972,558.416127,1.611856,1966.788218
std,0.955748,0.965921,0.691712,5.868725,90.676964,392.217403,3990.669241,0.960793,29.088642
min,1.000000,0.000000,0.000000,0.000000,3000.000000,0.000000,0.000000,0.000000,1196.000000
25%,2.000000,2.000000,1.000000,6.100000,3044.000000,122.000000,177.000000,1.000000,1960.000000
50%,3.000000,3.000000,1.000000,9.200000,3084.000000,126.000000,440.000000,2.000000,1970.000000
75%,3.000000,3.000000,2.000000,13.000000,3148.000000,129.940000,651.000000,2.000000,1975.000000
max,10.000000,20.000000,8.000000,48.100000,3977.000000,44515.000000,433014.000000,10.000000,2018.000000


**Building Your Model**
there are 4 steps for building and using model are:
1.Define:what type of model it would be , linear regression , Decision tree etc.
2.Fit: Capture patterns from provided data. This is the heart of modeling.
3.Predict: Just what it sounds like
4.Evaluate: Determine how accurate the model's predictions are.

In [137]:
# we will import DecisionTree model to train our model
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=1) # random_state=1 so each time model runs it remain same

model.fit(X, y)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",1
,"max_le

In [138]:
#test model 
print("making prediction for these 5 houses: ")
print(X.head())
print("The predictions are: ")
print(model.predict(X.head()).astype(int))
print(list(y.head().astype(int)))

making prediction for these 5 houses: 
   Rooms  Bedroom2  Bathroom  Distance  Postcode  BuildingArea  Landsize  Car  \
0      2       2.0       1.0       2.5    3067.0         126.0     202.0  1.0   
1      2       2.0       1.0       2.5    3067.0          79.0     156.0  0.0   
2      3       3.0       2.0       2.5    3067.0         150.0     134.0  0.0   
3      3       3.0       2.0       2.5    3067.0         126.0      94.0  1.0   
4      4       3.0       1.0       2.5    3067.0         142.0     120.0  2.0   

   YearBuilt  
0     1970.0  
1     1900.0  
2     1900.0  
3     1970.0  
4     2014.0  
The predictions are: 
[1480000 1035000 1465000  850000 1600000]
[1480000, 1035000, 1465000, 850000, 1600000]



# 3.Model Validation
Measure the performance of your model, so you can test and compare alternatives.
we will evaluate almost every model , in simple term we will check how well our model predicted , and we
compare predicted value to actual values.
we have many matrices to measure this but now we will focus on just MAE(Mean-Absolte-Error)
The prediction error for each house is:

error=actual−predicted
So, if a house cost $150,000 and you predicted it would cost $100,000 the error is $50,000.
in simple words,
On average, our predictions are off by about X.

In [139]:
#calculating mae
from sklearn.metrics import mean_absolute_error

predicted_prices = model.predict(X)
mae = mean_absolute_error(y,predicted_prices)
print(f"\nIn-Sample MAE: ${mae:,.2f}")


In-Sample MAE: $3,184.89


**in sample method**
we see that we use same sample for prediction that we use for training the model.
its same like giving student notes to study and student memorize them all , and then testing student from 
questions that are in notes, it is not the right way , right way is making test from other than that question that 
are related to notes but no exact.
in "in sample" method look we are doing this same error that's why we get "0.0" error.
The process of making test data is called" validation data".
best way to do that is spliting our main data

In [140]:
#coding it
from sklearn.model_selection import train_test_split
#spliting data into training data and testing data
X_train, X_test, y_train ,y_test = train_test_split(X, y ,test_size=0.2 , random_state=0)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
validation_mae = mean_absolute_error(y_test, y_pred)

print(f"Validation MAE (Unseen Data): ${validation_mae:,.2f}")

Validation MAE (Unseen Data): $245,221.73


# 4.Underfitting and Overfitting
Fine-tune your model for better performance.
**Experimenting With Different Models**
Example: we will control tree depth using max_leaf_nodes

In [141]:
def get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    return (mae)
size = [5, 50, 500, 1000,2000, 3000, 5000]
for max_leaf_nodes in size:
    my_mae = get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test)
    print("Max leaf nodes: %d \t\t Mean Absolute Error: %d" %(max_leaf_nodes,my_mae))
leaf_nodes = {leaf_size : get_mae(leaf_size,  X_train, X_test, y_train, y_test) for leaf_size in size}
best_leaf_node = min(leaf_nodes, key=leaf_nodes.get)
print("="*60,"\nbest leaf node: {}".format(best_leaf_node))


Max leaf nodes: 5 		 Mean Absolute Error: 361529
Max leaf nodes: 50 		 Mean Absolute Error: 245139
Max leaf nodes: 500 		 Mean Absolute Error: 227697
Max leaf nodes: 1000 		 Mean Absolute Error: 224212
Max leaf nodes: 2000 		 Mean Absolute Error: 233003
Max leaf nodes: 3000 		 Mean Absolute Error: 235479
Max leaf nodes: 5000 		 Mean Absolute Error: 241131
best leaf node: 1000


**modifying decision tree using best leaf node**

In [147]:
model = DecisionTreeRegressor(max_leaf_nodes=best_leaf_node, random_state=0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
validation_mae2 = mean_absolute_error(y_test,y_pred)
print(f"Validation MAE (Unseen Data) : \t before best leaf: ${validation_mae:,.2f} \t  after best leaf: ${validation_mae2:,.2f}")
print(f"Difference between model MAE on leaf nodes: {validation_mae - validation_mae2:,.2f}")

Validation MAE (Unseen Data) : 	 before best leaf: $245,221.73 	  after best leaf: $224,212.43
Difference between model MAE on leaf nodes: 21,009.30


# 5.Random Forests
Using a more sophisticated machine learning algorithm.

In [148]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

forest_model = RandomForestRegressor(random_state=1)
forest_model.fit(X_train,y_train)
forest_pred = forest_model.predict(X_test)
forest_mae=mean_absolute_error(y_test, forest_pred)
print(f"Validation MAE (Unseen Data): ${forest_mae:,.2f}")

Validation MAE (Unseen Data): $180,562.85


In [155]:
print(f"Best MAE from Decision tree '${validation_mae2:,.2f}' and MAE from RandomForest '${forest_mae:,.2f}'")

Best MAE from Decision tree '$224,212.43' and MAE from RandomForest '$180,562.85'
